# 12 Dashboard: duplicates
Numbers for the dashboard's **Duplicates** page. Runs every code cell of `99_dups.ipynb` (which itself replays `02_data_cleaning.ipynb` up to the dedupe), then saves its tables as small CSVs in `data/dashboard/`.

`99_dups` is not changed. It reads the raw `vehicles.csv` twice, so this takes a few minutes.

In [1]:
import json
import numpy as np
import pandas as pd
from carpricepredictor.shared import DASHBOARD_DIR

## Run 99_dups
Its variables (`df`, `pairs`, `kept`, `posts`, `show_car`, ...) end up in this notebook. The loop uses its own names (`dup_cell`, `dup_src`), because `99_dups` runs a loop of its own with `cell` / `src`.

One line of `99_dups` is swapped while replaying it: the placeholder-VIN check (`real_vin`) uses the regex `(.)\1+`, and pandas 3's default Arrow strings can't run the `\1` backreference. The same check runs on Python strings (`.astype(object)`), whose regex engine supports it. `99_dups` itself is left as it is.

In [2]:
# 99_dups' placeholder-VIN check, and the same check on Python strings (Arrow strings can't run \1)
VIN_CHECK = r'~vin.str.fullmatch(r"(.)\1+", na=False)'
VIN_CHECK_FIXED = r'~vin.astype(object).str.fullmatch(r"(.)\1+", na=False)'

dup_cells = [c for c in json.load(open("99_dups.ipynb"))["cells"] if c["cell_type"] == "code"]
assert sum(VIN_CHECK in "".join(c["source"]) for c in dup_cells) == 1, "the real_vin line in 99_dups changed"
for dup_cell in dup_cells:
    dup_src = "".join(dup_cell["source"]).replace(VIN_CHECK, VIN_CHECK_FIXED)
    exec(dup_src)

357,696 rows before the dedupe, 152,311 removed (42.6%)
dealer listings among them: 97%
after the dedupe, 8,170 rows still share a VIN with another row (3,887 cars)
same post scraped twice: 0 rows
dealer listings: 97% of removed rows | Carvana alone: 35%
2020 honda passport | VIN 5FNYF8H01LB009820 | 4 posts in 4 regions, 3 states
first url: https://ventura.craigslist.org/ctd/d/atlanta-2020-honda-passport-elite-sport/7301963717.html
2021 chevrolet trailblazer | VIN KL79MMS2XMB026016 | 4 posts in 4 regions, 2 states
first url: https://harrisburg.craigslist.org/ctd/d/darlington-2021-chevrolet-trailblazer/7303624197.html
2020 chevrolet silverado 3500 | VIN 1GB3WREY1LF301140 | 4 posts in 4 regions, 2 states
first url: https://charlotte.craigslist.org/cto/d/greensboro-2020-chevrolet-silverado/7316414731.html
2017 ford expedition | VIN 1FMJU1JT1HEA52352 | 261 posts in 261 regions, 38 states
first url: https://lascruces.craigslist.org/ctd/d/las-cruces-2017-ford-expedition-xlt/7311766797.html
p

## Headline numbers
One row per number, so the page can show them as cards and in its text.

In [3]:
# 99_dups reuses the name `vin` in a later loop, so the VIN column is rebuilt here
kept_vins = kept["VIN"].str.upper()
out_summary = pd.Series({
    "rows before the dedupe": len(df),
    "removed": len(removed),
    "kept": len(kept),
    "removed share": len(removed) / len(df),
    "dealer share of removed": pairs["dealer"].mean(),
    "carvana share of removed": pairs["carvana"].mean(),
    "same post scraped twice": int(same_post.sum()),
    "rows still sharing a VIN": int(repost.sum()),
    "cars still sharing a VIN": int(kept_vins[repost].nunique()),
}).rename_axis("metric").rename("value").reset_index()
out_summary

,metric,value
0,rows before the dedupe,357696.000000
1,removed,152311.000000
2,kept,205385.000000
3,removed share,0.425811
4,dealer share of removed,0.970652
5,carvana share of removed,0.354603
6,same post scraped twice,0.000000
7,rows still sharing a VIN,8170.000000
8,cars still sharing a VIN,3887.000000


## Evidence and why
- **evidence**: was each removed row really the same car (section 3 of `99_dups`), with the dealer share per group
- **why**: why the copy exists (section 6): scraped twice, reposted in the same region, another region, another state

In [4]:
by_evidence = pairs.groupby("evidence")
out_evidence = pd.DataFrame({
    "rows": by_evidence.size(),
    "share": by_evidence.size() / len(pairs),
    "dealer": by_evidence["dealer"].mean(),
}).sort_values("rows", ascending=False).rename_axis("evidence").reset_index()

by_why = pairs.groupby("why")
out_why = pd.DataFrame({
    "rows": by_why.size(),
    "share": by_why.size() / len(pairs),
    "dealer": by_why["dealer"].mean(),
    "carvana": by_why["carvana"].mean(),
    "median_days_apart": days_apart.groupby(pairs["why"]).median(),
}).sort_values("rows", ascending=False).rename_axis("why").reset_index()
display(out_evidence.round(3))
out_why.round(3)

,evidence,rows,share,dealer
0,same VIN: same car,120744,0.793,0.996
1,"no VIN to compare, same description",19790,0.130,0.873
2,"no VIN to compare, different description",11717,0.077,0.879
3,different VIN: different cars,60,0.000,0.967


,why,rows,share,dealer,carvana,median_days_apart
0,posted in another state,86819,0.570,0.990,0.583,0.209
1,"posted in another region, same state",34664,0.228,0.966,0.098,0.204
2,reposted in the same region,30828,0.202,0.922,0.001,7.147


## Posts per car
How many times each car (real VIN) was posted before the dedupe.

In [5]:
posts_per_vin = posts.groupby("VIN").size()
out_posts = posts_per_vin.value_counts().sort_index().rename_axis("posts").rename("cars").reset_index()
out_posts.head(12)

,posts,cars
0,1,63034
1,2,13957
2,3,5926
3,4,3740
4,5,2280
5,6,1484
6,7,920
7,8,623
8,9,499
9,10,316


## Example cars
The three cars from section 7 of `99_dups` (Carvana, a local dealer, a private owner) and the most posted car, one row per post. `usual_price`: the post has the car's most common price (a post at another price is how a repost slips past the dedupe).

In [6]:
EXAMPLES = {
    "5FNYF8H01LB009820": "Carvana",
    "KL79MMS2XMB026016": "local dealer",
    "1GB3WREY1LF301140": "private owner",
    posts.groupby("VIN")["region"].nunique().idxmax(): "most posted car",
}

def car_title(vin):
    car = posts[posts["VIN"] == vin].iloc[0]
    return f"{car['year']} {car['manufacturer']} {car['model']}"

out_examples = pd.concat([
    show_car(v).assign(vin=v, kind=kind, car=car_title(v)) for v, kind in EXAMPLES.items()
], ignore_index=True)
out_examples["usual_price"] = out_examples["price"] == out_examples.groupby("vin")["price"].transform(lambda p: p.mode().iloc[0])
out_examples.groupby(["kind", "car"], sort=False)["usual_price"].agg(["size", "sum"])

2020 honda passport | VIN 5FNYF8H01LB009820 | 4 posts in 4 regions, 3 states
first url: https://ventura.craigslist.org/ctd/d/atlanta-2020-honda-passport-elite-sport/7301963717.html
2021 chevrolet trailblazer | VIN KL79MMS2XMB026016 | 4 posts in 4 regions, 2 states
first url: https://harrisburg.craigslist.org/ctd/d/darlington-2021-chevrolet-trailblazer/7303624197.html
2020 chevrolet silverado 3500 | VIN 1GB3WREY1LF301140 | 4 posts in 4 regions, 2 states
first url: https://charlotte.craigslist.org/cto/d/greensboro-2020-chevrolet-silverado/7316414731.html
2017 ford expedition | VIN 1FMJU1JT1HEA52352 | 261 posts in 261 regions, 38 states
first url: https://lascruces.craigslist.org/ctd/d/las-cruces-2017-ford-expedition-xlt/7311766797.html


,,size,sum
kind,car,,
Carvana,2020 honda passport,4,4
local dealer,2021 chevrolet trailblazer,4,4
private owner,2020 chevrolet silverado 3500,4,3
most posted car,2017 ford expedition,261,261


## Same text in every post?
Section 8 of `99_dups`: per kind of seller, the share of cars whose model text / ad text is identical in every post, and how similar the ads are (difflib, 1.0 = identical).

In [7]:
similarity = pd.DataFrame(scores).groupby("kind")["similarity"]
out_text = summary.join(similarity.median().rename("median_similarity")).join(similarity.size().rename("compared_pairs"))
out_text = out_text.rename_axis("kind").reset_index()
out_text.round(3)

,kind,same model text,same ad text,cars,median_similarity,compared_pairs
0,Carvana,1.00,0.00,5738,0.994,147
1,other dealer,0.94,0.30,25757,0.995,515
2,private owner,0.83,0.52,483,0.966,5


## What the dedupe misses, and the doubtful rows
- **reposts**: cars that still share a real VIN after the dedupe (section 5), and how often each column differs between their copies
- **wrong**: removed rows whose VIN differs from the kept row (different cars with identical values)
- **unsure**: removed rows with no VIN to compare and a different description

In [8]:
out_reposts = ((groups[["price", "odometer", "state", "condition"]].nunique() > 1).mean()
               .rename_axis("column").rename("share of cars where it differs").reset_index())

out_wrong = wrong[["price", "year", "manufacturer", "model", "odometer", "state_removed", "state_kept",
                   "VIN_removed", "VIN_kept"]].drop_duplicates(["VIN_removed", "VIN_kept"]).head(6)
out_unsure = unsure.assign(
    ad_removed=unsure["description_removed"].str[:80],
    ad_kept=unsure["description_kept"].str[:80],
)[["price", "year", "manufacturer", "model", "odometer", "ad_removed", "ad_kept"]].head(6)
out_reposts

,column,share of cars where it differs
0,price,0.551839
1,odometer,0.117571
2,state,0.241060
3,condition,0.125547


## Save

In [9]:
DASHBOARD_DIR.mkdir(exist_ok=True)
out = {
    "dup_summary.csv": out_summary,
    "dup_evidence.csv": out_evidence,
    "dup_why.csv": out_why,
    "dup_posts_per_car.csv": out_posts,
    "dup_examples.csv": out_examples,
    "dup_text.csv": out_text,
    "dup_reposts.csv": out_reposts,
    "dup_wrong.csv": out_wrong,
    "dup_unsure.csv": out_unsure,
}
for name, table in out.items():
    table.to_csv(DASHBOARD_DIR / name, index=False)
    print(f"{name:22} {len(table):>6,} rows")

dup_summary.csv             9 rows
dup_evidence.csv            4 rows
dup_why.csv                 3 rows
dup_posts_per_car.csv     136 rows
dup_examples.csv          273 rows
dup_text.csv                3 rows
dup_reposts.csv             4 rows
dup_wrong.csv               6 rows
dup_unsure.csv              6 rows
